# S02E01 — Categorize

**Zadanie:** Sklasyfikować 10 towarów jako `DNG` lub `NEU` przy użyciu promptu mieszczącego się w **100 tokenach**.  
Wyjątek: kasety/paliwo do reaktora → zawsze `NEU` (żeby uniknąć kontroli).

**Techniki:** inżynieria promptów, zarządzanie kontekstem, prompt caching, iteracyjny agent

**Budżet:** 1.5 PP na całe zadanie (10 zapytań)

In [ ]:
import os, csv, io, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
CSV_URL    = f"https://hub.ag3nts.org/data/{API_KEY}/categorize.csv"

print("Konfiguracja OK.")

In [ ]:
def fetch_items() -> list[dict]:
    """Pobierz świeżą listę towarów z Hub-u."""
    resp = requests.get(CSV_URL)
    resp.raise_for_status()
    reader = csv.DictReader(io.StringIO(resp.text))
    items  = list(reader)
    print(f"Pobrano {len(items)} towarów")
    return items


def send_prompt(prompt: str) -> dict:
    """Wyślij prompt klasyfikujący do Hub-u i zwróć odpowiedź."""
    resp = requests.post(VERIFY_URL, json={
        "apikey": API_KEY,
        "task": "categorize",
        "answer": {"prompt": prompt}
    })
    return resp.json()


def reset_task():
    """Zresetuj licznik zadania."""
    return send_prompt("reset")


print("Funkcje gotowe.")

In [ ]:
# Zaprojektuj prompt klasyfikujący za pomocą Claude
items = fetch_items()
sample_items = items[:3]  # pokaż próbkę

design_resp = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Napisz prompt klasyfikujący towary jako DNG (niebezpieczne) lub NEU (neutralne).

WYMAGANIA:
1. Prompt + dane towaru muszą mieścić się w 100 tokenach łącznie.
2. Prompt zwraca TYLKO słowo DNG lub NEU — nic więcej.
3. Kasety reaktorowe, paliwo jądrowe, elementy reaktora → ZAWSZE NEU (wyjątek bezpieczeństwa).
4. Pisz po angielsku (mniej tokenów niż po polsku).
5. Dane towaru będą na końcu: ID: {{id}} DESC: {{description}}

Przykładowe towary:
{chr(10).join(f'ID: {i["id"]} DESC: {i["description"]}' for i in sample_items)}

Zwróć TYLKO prompt (bez wyjaśnień). Używaj {{id}} i {{description}} jako placeholderów."""
    }]
)

PROMPT_TEMPLATE = design_resp.content[0].text.strip()
print("Zaprojektowany prompt:")
print(PROMPT_TEMPLATE)

In [ ]:
# Sprawdź liczbę tokenów dla najdłuższego towaru
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

max_tokens_count = 0
for item in items:
    filled = PROMPT_TEMPLATE.replace("{id}", item["id"]).replace("{description}", item["description"])
    n = len(enc.encode(filled))
    max_tokens_count = max(max_tokens_count, n)
    print(f"  {item['id']}: {n} tokenów")

print(f"\nMaksimum: {max_tokens_count} tokenów (limit: 100)")
if max_tokens_count > 100:
    print("UWAGA: Prompt za długi! Skróć go.")
else:
    print("OK — mieścimy się w limicie.")

In [ ]:
# Iteracyjna klasyfikacja z automatycznym poprawianiem
MAX_ATTEMPTS = 5

for attempt in range(MAX_ATTEMPTS):
    print(f"\n=== Próba {attempt+1} ===")

    # Zawsze pobieraj świeże dane (lista zmienia się co kilka minut)
    items = fetch_items()

    errors    = []
    got_flag  = False

    for item in items:
        filled = PROMPT_TEMPLATE.replace("{id}", item["id"]).replace("{description}", item["description"])
        result = send_prompt(filled)
        code   = result.get("code", -1)
        msg    = result.get("message", "")

        print(f"  {item['id']}: code={code} | {msg[:80]}")

        if "FLG:" in str(result):
            import re
            flags = re.findall(r'\{FLG:[^}]+\}', str(result))
            print(f"\nFLAGA: {flags}")
            got_flag = True
            break

        if code != 0:
            errors.append({"item": item, "error": msg})
            break  # Hub resetuje przy błędzie — zaczynamy od nowa

    if got_flag:
        break

    if errors:
        print(f"\nBłędy: {errors}")
        # Poproś Claude o naprawę promptu
        fix_resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            messages=[{
                "role": "user",
                "content": f"""Popraw prompt klasyfikujący. Obecny prompt:\n{PROMPT_TEMPLATE}\n\nBłędy:\n{errors}\n\nZwróć TYLKO poprawiony prompt."""
            }]
        )
        PROMPT_TEMPLATE = fix_resp.content[0].text.strip()
        print(f"Nowy prompt: {PROMPT_TEMPLATE}")
        reset_task()

print("\nZakończono.")